In [1]:
#CODE LOAD DATA
!pip install pandas numpy lightgbm shap matplotlib seaborn plotly prophet -q

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Import thành công")

data_path = r"C:\DATATHON ROUND1" #THAY DATA PATH PHÙ HỢP VÀO
folder = Path(data_path)

files = {
        'products.csv': {},
        'customers.csv': {'parse_dates': ['signup_date']},
        'promotions.csv': {'parse_dates': ['start_date', 'end_date']},
        'geography.csv': {},
        'orders.csv': {'parse_dates': ['order_date']},
        'order_items.csv': {},
        'payments.csv': {},
        'shipments.csv': {'parse_dates': ['ship_date', 'delivery_date']},
        'returns.csv': {'parse_dates': ['return_date']},
        'reviews.csv': {'parse_dates': ['review_date']},
        'sales.csv': {'parse_dates': ['Date']},
        'sample_submission.csv': {'parse_dates': ['Date']},
        'inventory.csv': {'parse_dates': ['snapshot_date']},
        'web_traffic.csv': {'parse_dates': ['date']},
}

data = {}
for filename, kwargs in files.items():
    df = pd.read_csv(folder / filename, **kwargs)
    data[filename.replace('.csv', '')] = df
    print(f" {filename:<20} | Shape: {df.shape}")



Import thành công
 products.csv         | Shape: (2412, 8)
 customers.csv        | Shape: (121930, 7)
 promotions.csv       | Shape: (50, 10)
 geography.csv        | Shape: (39948, 4)
 orders.csv           | Shape: (646945, 8)
 order_items.csv      | Shape: (714669, 7)
 payments.csv         | Shape: (646945, 4)
 shipments.csv        | Shape: (566067, 4)
 returns.csv          | Shape: (39939, 7)
 reviews.csv          | Shape: (113551, 7)
 sales.csv            | Shape: (3833, 3)
 sample_submission.csv | Shape: (548, 3)
 inventory.csv        | Shape: (60247, 17)
 web_traffic.csv      | Shape: (3652, 7)


In [2]:
orders = data['orders'] # Thêm dòng này trước khi tính toán

multi_order_cust = orders['customer_id'].value_counts()
multi_order_cust = multi_order_cust[multi_order_cust > 1].index

q1_df = orders[orders['customer_id'].isin(multi_order_cust)].sort_values(['customer_id', 'order_date'])
q1_df['gap'] = q1_df.groupby('customer_id')['order_date'].diff().dt.days

median_gap = q1_df['gap'].dropna().median()
print(f"Q1 - Trung vị inter-order gap: {median_gap} ngày")

Q1 - Trung vị inter-order gap: 144.0 ngày


In [3]:
products = pd.read_csv(folder / 'products.csv')

products['gross_margin_rate'] = (products['price'] - products['cogs']) / products['price']
avg_margin_by_segment = products.groupby('segment')['gross_margin_rate'].mean().sort_values(ascending=False)

print("Q2 - Tỷ suất lợi nhuận gộp trung bình theo phân khúc:")
print(avg_margin_by_segment)

Q2 - Tỷ suất lợi nhuận gộp trung bình theo phân khúc:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin_rate, dtype: float64


In [4]:
returns = pd.read_csv(folder / 'returns.csv')

returns_products = returns.merge(products[['product_id', 'category']], on='product_id')

streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']
top_reason = streetwear_returns['return_reason'].value_counts()

print("Q3 - Lý do trả hàng Streetwear phổ biến nhất:")
print(top_reason)

Q3 - Lý do trả hàng Streetwear phổ biến nhất:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


In [5]:
web = data['web_traffic']
lowest_bounce_source = web.groupby('traffic_source')['bounce_rate'].mean().sort_values()

print("Q4 - Nguồn truy cập có tỷ lệ thoát thấp nhất:")
print(lowest_bounce_source.head(1))

Q4 - Nguồn truy cập có tỷ lệ thoát thấp nhất:
traffic_source
email_campaign    0.004458
Name: bounce_rate, dtype: float64


In [6]:
order_items = data['order_items']

total_items = len(order_items)
promo_items = order_items['promo_id'].notnull().sum()
promo_rate = (promo_items / total_items) * 100

print(f"Q5 - Tỷ lệ dòng hàng có khuyến mãi: {promo_rate:.2f}%")

Q5 - Tỷ lệ dòng hàng có khuyến mãi: 38.66%


In [7]:
customers = pd.read_csv(folder / 'customers.csv')

cust_orders = orders.groupby('customer_id').size().reset_index(name='order_count')
cust_data = customers.merge(cust_orders, on='customer_id', how='left').fillna({'order_count': 0})

age_group_analysis = cust_data[cust_data['age_group'].notnull()].groupby('age_group')['order_count'].mean().sort_values(ascending=False)

print("Q6 - Số đơn hàng trung bình theo nhóm tuổi:")
print(age_group_analysis)

Q6 - Số đơn hàng trung bình theo nhóm tuổi:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64


In [8]:
geography = pd.read_csv(folder / 'geography.csv')
order_items = data['order_items']

order_items['line_revenue'] = order_items['quantity'] * order_items['unit_price']
order_revenue = order_items.groupby('order_id')['line_revenue'].sum().reset_index()

order_region_map = orders.merge(geography[['zip', 'region']], on='zip', how='left')

final_geo_rev = order_region_map.merge(order_revenue, on='order_id')

q7_result = final_geo_rev.groupby('region')['line_revenue'].sum().sort_values(ascending=False)

print("Q7 - Tổng doanh thu theo vùng:")
print(q7_result)

Q7 - Tổng doanh thu theo vùng:
region
East       7.637533e+09
Central    4.941908e+09
West       3.851035e+09
Name: line_revenue, dtype: float64


In [9]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']
top_payment_cancelled = cancelled_orders['payment_method'].value_counts()

print("Q8 - Phương thức thanh toán của đơn hàng bị hủy:")
print(top_payment_cancelled)

Q8 - Phương thức thanh toán của đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


In [10]:
order_items = data['order_items']

items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id')
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id')

sales_count = items_with_size['size'].value_counts()
return_count = returns_with_size['size'].value_counts()

return_rate_size = (return_count / sales_count).sort_values(ascending=False)
print("Q9 - Tỷ lệ trả hàng theo kích thước:")
print(return_rate_size)

Q9 - Tỷ lệ trả hàng theo kích thước:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64


In [11]:
payments = pd.read_csv(folder / 'payments.csv')

avg_payment_installments = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

print("Q10 - Giá trị thanh toán trung bình theo kỳ trả góp:")
print(avg_payment_installments)

Q10 - Giá trị thanh toán trung bình theo kỳ trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
